# Complexité algorithmique, Big O et programmation dynamique en Python

## Objectifs

- comprendre taille d’entrée, opération et fonction de coût ;
- comprendre puissances et logarithmes avant `O(log n)` ;
- reconnaître les complexités courantes dans du Python ;
- choisir une structure de données adaptée ;
- comparer recherche linéaire et dichotomique ;
- comprendre récursion, mémoïsation et programmation dynamique ;
- distinguer analyse théorique, chronométrage et profiling.

Ce cours développe le document [`complexity.pdf`](../../ressources/source/complexity.pdf) avec des explications et exercices pour débutants.

## Comment utiliser le notebook

Exécutez les cellules dans l’ordre. Avant une démonstration, essayez de prévoir le résultat. Pour chaque exercice, écrivez d’abord le raisonnement en français : Big O décrit une idée de croissance avant d’être un symbole.

Les temps mesurés varieront selon l’ordinateur. C’est normal : la mesure dépend de la machine, Big O décrit l’évolution lorsque l’entrée grandit.

## La question générale

![Entrée, algorithme, coût et Big O](../../ressources/illustrations/jour_01/00_pipeline_big_o.png)

Un **algorithme** est une suite finie d’étapes transformant une entrée en résultat. La lettre **n** représente la taille de l’entrée : nombre d’éléments, caractères, sommets, pixels, etc.

Big O ne donne pas une durée exacte. Il classe la croissance du travail lorsque `n` augmente.

# 1. Bases mathématiques utiles

Nous n’avons besoin ni de dérivées ni de preuves avancées. Il faut comprendre une fonction, comparer des rythmes de croissance, reconnaître une puissance et interpréter un logarithme.

## Fonction de coût et opération élémentaire

$T(n)=3n+5$ signifie que notre modèle compte environ `3 × n + 5` opérations pour une entrée de taille `n`.

Une **opération élémentaire** est une étape choisie comme unité : comparaison, addition, lecture d’une case. Le modèle simplifie la réalité, mais permet de comparer des algorithmes sans dépendre d’un ordinateur précis.

Si `n` double, `3n+5` est presque doublé, `n²` est quadruplé et `2ⁿ` est mis au carré.

In [ ]:
import math
import timeit
import cProfile
import pstats
import io
from bisect import bisect_left
from collections import deque
from functools import cache
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
print("Environnement prêt")

In [ ]:
print(f"{'n':>10} {'3n + 5':>12} {'n²':>15}")
for n in [10, 100, 1_000, 10_000]:
    print(f"{n:>10,} {3*n+5:>12,} {n*n:>15,}")

## Puissance et exponentielle

Dans $n^2$, `n` est la base : `n × n`. Doubler `n` quadruple `n²`.

Dans $2^n$, `n` est l’exposant : on multiplie `2` par lui-même `n` fois. Ajouter `1` à `n` double le résultat. Une croissance exponentielle finit donc par dépasser très vite les croissances polynomiales.

## Qu’est-ce qu’un logarithme ?

![Divisions expliquant log2](../../ressources/illustrations/jour_01/01_comprendre_logarithme.png)

Le logarithme répond à la question inverse d’une puissance : $2^5=32$, donc $\log_2(32)=5$.

Pour un algorithme qui divise son problème par deux, $\log_2(n)$ compte approximativement le nombre d’étapes avant d’atteindre `1`. Pour 100 éléments : `100 → 50 → 25 → 12 → 6 → 3 → 1`, soit au plus 7 divisions.

In [ ]:
for n in [8, 16, 32, 100, 1_000, 1_000_000, 1_000_000_000]:
    print(f"n={n:>13,} | log2={math.log2(n):>6.2f} | plafond={math.ceil(math.log2(n)):>2}")

### Pourquoi la base disparaît-elle dans Big O ?

Changer de base multiplie le logarithme par une constante. Big O ignore les facteurs constants : on écrit `O(log n)`. Pour la dichotomie, la base 2 reste la plus intuitive puisque chaque étape garde une moitié.

## À vous de jouer — compter les divisions

Pour `8`, `64`, `1 024` et `1 000 000`, calculez le plafond de $\log_2(n)$ et expliquez ce que représente le résultat.

In [ ]:
for value in [8, 64, 1_024, 1_000_000]:
    print(f"n={value:>9,} → {math.ceil(math.log2(value)):>2} divisions au maximum")

## Familles de croissance

![Courbes de complexité](../../ressources/illustrations/jour_01/02_croissance_complexites.png)

La courbe exponentielle est plafonnée sur l’image, car elle sortirait immédiatement du graphique. Une complexité élevée n’est pas automatiquement interdite : certains problèmes restent utilisables pour de petits `n`. Elle indique surtout jusqu’où l’approche pourra monter.

# 2. Notation Big O

Big O fournit une **borne asymptotique supérieure**. Pour débuter, retenez : on garde la partie du coût qui explique sa croissance lorsque `n` devient très grand.

![Simplification Big O](../../ressources/illustrations/jour_01/03_simplifier_big_o.png)

- $3n+5$ devient `O(n)` ;
- $n²+3n+100$ devient `O(n²)` ;
- `7` devient `O(1)`.

Les constantes influencent le temps réel, mais pas la famille de croissance.

## Complexités courantes

| Complexité | Nom | Intuition | Exemple |
|---|---|---|---|
| `O(1)` | constante | même nombre d’étapes | `values[0]` |
| `O(log n)` | logarithmique | diviser le problème | dichotomie |
| `O(n)` | linéaire | visiter les éléments | somme |
| `O(n log n)` | quasi linéaire | diviser puis traiter | tri efficace |
| `O(n²)` | quadratique | traiter des paires | deux boucles |
| `O(2ⁿ)` | exponentielle | explorer des choix | Fibonacci naïf, borne simple |
| `O(n!)` | factorielle | tester tous les ordres | permutations |

## Big O n’est pas synonyme de pire cas

On analyse souvent le **pire cas** pour obtenir une garantie, mais on peut décrire le meilleur, le moyen ou le pire cas.

Recherche linéaire : meilleur cas `O(1)` si la cible est au début ; pire cas `O(n)` si elle est absente ou à la fin ; cas moyen `O(n)` car les constantes sont ignorées.

## Temps et mémoire

La complexité temporelle décrit le nombre d’étapes. La complexité spatiale auxiliaire décrit la mémoire supplémentaire, sans recompter l’entrée fournie. Utiliser un `set` peut accélérer une recherche au prix de mémoire supplémentaire.

## Lire la complexité dans le code

![Règles de lecture](../../ressources/illustrations/jour_01/04_regles_lecture_code.png)

Les blocs successifs s’additionnent, les boucles imbriquées multiplient leurs nombres d’itérations, et une réduction répétée par un facteur fixe produit souvent un logarithme.

In [ ]:
def example_a(values):
    return values[0]                 # O(1)

def example_b(values):
    total = 0
    for value in values:             # n passages
        total += value
    return total                     # O(n)

def example_c(values):
    total = 0
    for left in values:
        for right in values:
            total += left == right
    return total                     # O(n²)

def example_d(n):
    steps = 0
    while n > 1:
        n //= 2
        steps += 1
    return steps                     # O(log n)

print(example_d(1_000_000))

## À vous de jouer — classer les extraits

Associez `example_a`, `example_b`, `example_c` et `example_d` à `O(1)`, `O(log n)`, `O(n)` ou `O(n²)`.

In [ ]:
answers = {"A": "O(1)", "B": "O(n)", "C": "O(n²)", "D": "O(log n)"}
answers

## Même Big O, temps réel différent

`1000n` et `2n` sont tous deux `O(n)`, mais le second est généralement plus rapide. Big O ne remplace pas la mesure. Inversement, un code `O(n²)` très simple peut gagner sur de minuscules entrées face à un algorithme sophistiqué.

# 3. Structures natives de Python

Une instruction courte n’est pas nécessairement une opération constante. Il faut comprendre l’organisation de la structure utilisée.

![Structures Python](../../ressources/illustrations/jour_01/05_structures_python.png)

`set` et `dict` utilisent une table de hachage : recherche `O(1)` en moyenne, `O(n)` dans un pire cas théorique. Une `list` permet l’accès par indice en `O(1)`, mais insérer au début décale les éléments. Une `deque` ajoute ou retire efficacement aux extrémités.

## Le piège de `in`

Dans une liste, Python compare les éléments jusqu’à trouver la valeur : `O(n)` au pire. Dans un set : `O(1)` en moyenne. Construire le set coûte cependant `O(n)` en temps et en espace ; il est surtout rentable pour des recherches répétées.

In [ ]:
membership_values = list(range(100_000))
membership_set = set(membership_values)
missing = -1
list_time = timeit.timeit(lambda: missing in membership_values, number=100)
set_time = timeit.timeit(lambda: missing in membership_set, number=100)
print(f"liste : {list_time:.6f} s | set : {set_time:.6f} s")

## Retirer au début

`list.append()` et `list.pop()` à la fin coûtent `O(1)` **amorti** : quelques agrandissements coûtent cher, mais leur moyenne reste constante. `list.pop(0)` coûte `O(n)` à cause des décalages. Pour une file FIFO, `deque.append()` et `deque.popleft()` coûtent `O(1)`.

In [ ]:
def remove_list_front():
    values = list(range(20_000))
    for _ in range(300): values.pop(0)

def remove_deque_front():
    values = deque(range(20_000))
    for _ in range(300): values.popleft()

print("list :", timeit.timeit(remove_list_front, number=30))
print("deque:", timeit.timeit(remove_deque_front, number=30))

## Tableau simplifié

| Opération | `list` | `set` | `dict` | `deque` |
|---|---:|---:|---:|---:|
| indice | `O(1)` | — | — | `O(n)` |
| valeur/clé | `O(n)` | `O(1)` moyen | `O(1)` moyen | `O(n)` |
| ajout fin | `O(1)` amorti | `O(1)` moyen | `O(1)` moyen | `O(1)` |
| ajout/retrait début | `O(n)` | — | — | `O(1)` |

## À vous de jouer — choisir la structure

Choisissez entre `list`, `set`, `dict` et `deque` pour des tests de présence répétés, une file FIFO, un accès fréquent par indice et l’association d’un identifiant à une fiche.

In [ ]:
choices = {
    "présences répétées": "set",
    "file FIFO": "deque",
    "accès fréquent par indice": "list",
    "associer identifiant et fiche": "dict",
}
choices

# 4. Recherche linéaire et dichotomique

Les deux algorithmes cherchent la même chose. La dichotomie possède une information supplémentaire : la collection est triée.

## Recherche linéaire

![Recherche linéaire](../../ressources/illustrations/jour_01/06_recherche_lineaire.png)

Elle inspecte les éléments un à un et fonctionne avec des données triées ou non. Au pire : temps O(n), espace auxiliaire O(1).

In [ ]:
def recherche_lineaire(values, target):
    steps = 0
    for index, value in enumerate(values):
        steps += 1
        if value == target:
            return index, steps
    return -1, steps

demo_values = [4, 9, 13, 21, 28, 35, 42, 57]
print(recherche_lineaire(demo_values, 57))
print(recherche_lineaire(demo_values, 99))

## Recherche dichotomique

![Recherche dichotomique](../../ressources/illustrations/jour_01/07_recherche_dichotomique.png)

Elle compare la cible à l’élément central puis élimine la moitié impossible. Temps O(log n), espace O(1) dans la version itérative. Sans tri, cette élimination n’est pas justifiée.

In [ ]:
def recherche_dichotomique(values, target):
    low, high, steps = 0, len(values) - 1, 0
    while low <= high:
        steps += 1
        middle = (low + high) // 2
        if values[middle] == target:
            return middle, steps
        if values[middle] < target:
            low = middle + 1
        else:
            high = middle - 1
    return -1, steps

print(recherche_dichotomique(demo_values, 57))
print(recherche_dichotomique(demo_values, 99))

## L’écart quand n grandit

| Éléments | Linéaire, pire cas | Dichotomie, environ |
|---:|---:|---:|
| 10 | 10 | 4 |
| 1 000 | 1 000 | 10 |
| 1 000 000 | 1 000 000 | 20 |
| 1 000 000 000 | 1 000 000 000 | 30 |

Multiplier n par mille ajoute seulement environ dix étapes à la dichotomie, car 2 puissance 10 vaut 1024.

In [ ]:
print(f"{'n':>15} {'linéaire':>15} {'dichotomie':>15}")
for n in [10, 1_000, 1_000_000, 1_000_000_000]:
    print(f"{n:>15,} {n:>15,} {math.ceil(math.log2(n)):>15}")

## Faut-il trier pour chercher une seule fois ?

Trier coûte généralement O(n log n). Pour une seule recherche, trier puis chercher est plus coûteux qu’une recherche linéaire O(n). La dichotomie est rentable lorsque les données sont déjà triées ou lorsque le tri est amorti sur de nombreuses recherches.

Le module bisect trouve une position en O(log n), mais insérer ensuite dans une liste reste O(n) à cause des décalages.

In [ ]:
position = bisect_left(demo_values, 28)
found = position < len(demo_values) and demo_values[position] == 28
print("position :", position, "trouvé :", found)

## À vous de jouer — dichotomie comptée

Écrivez une fonction qui renvoie le couple indice et nombre d’étapes. Testez une valeur présente puis absente dans un million d’entiers pairs.

In [ ]:
def recherche_dichotomique_exercice(values, target):
    low, high, steps = 0, len(values) - 1, 0
    while low <= high:
        steps += 1
        middle = (low + high) // 2
        if values[middle] == target:
            return middle, steps
        if values[middle] < target:
            low = middle + 1
        else:
            high = middle - 1
    return -1, steps

sorted_values = list(range(0, 2_000_000, 2))
print(recherche_dichotomique_exercice(sorted_values, 1_999_998))
print(recherche_dichotomique_exercice(sorted_values, 1_999_999))

# 5. Récursion et programmation dynamique

La programmation dynamique évite de recalculer les mêmes sous-problèmes. C’est une stratégie de réutilisation, pas simplement un type de boucle.

## Qu’est-ce que la récursion ?

Une fonction récursive s’appelle sur un problème plus petit. Elle nécessite un cas de base et un appel qui s’en rapproche. Chaque appel en attente occupe la pile d’appels : une profondeur n utilise O(n) d’espace, même sans liste explicite.

## Fibonacci naïf

F(0)=0, F(1)=1 et F(n)=F(n-1)+F(n-2).

![Arbre Fibonacci](../../ressources/illustrations/jour_01/08_arbre_fibonacci.png)

La version naïve crée presque deux branches par appel. O(2 puissance n) est une borne simple ; une analyse plus précise donne environ O(phi puissance n), avec phi proche de 1,618. La profondeur maximale vaut n, donc l’espace de pile est O(n).

In [ ]:
naive_calls = 0

def fibo_naive(n):
    global naive_calls
    naive_calls += 1
    if n <= 1:
        return n
    return fibo_naive(n - 1) + fibo_naive(n - 2)

naive_calls = 0
print("F(20) =", fibo_naive(20))
print("nombre d'appels =", naive_calls)

## Mémoïsation : approche Top-Down

![Mémoïsation](../../ressources/illustrations/jour_01/09_memoisation.png)

Le cache conserve le résultat d’un appel. Un sous-problème déjà rencontré est lu au lieu d’être recalculé. Il ne reste que n+1 sous-problèmes distincts : temps O(n), espace O(n).

In [ ]:
def fibo_memo(n, memo=None):
    if memo is None:
        memo = {}
    if n <= 1:
        return n
    if n not in memo:
        memo[n] = fibo_memo(n - 1, memo) + fibo_memo(n - 2, memo)
    return memo[n]

print("F(100) =", fibo_memo(100))

### Pourquoi utiliser memo=None ?

Un dictionnaire placé directement comme valeur par défaut serait partagé entre les appels. Le motif sûr est memo=None, puis memo={} dans la fonction. Le test memo is None ne confond pas une absence de cache et un cache vide valide.

## Mémoïsation native avec functools.cache

Le décorateur cache mémorise automatiquement une fonction lorsque ses arguments peuvent servir de clés. Le code est plus court, mais la mémoire est toujours consommée.

In [ ]:
@cache
def fibo_cached(n):
    if n <= 1:
        return n
    return fibo_cached(n - 1) + fibo_cached(n - 2)

print("F(100) =", fibo_cached(100))
print(fibo_cached.cache_info())

## Bottom-Up avec une table

On part de F(0) et F(1), puis on monte jusqu’à F(n). La boucle évite la récursion. Temps O(n), espace O(n).

In [ ]:
def fibo_bottom_up(n):
    if n <= 1:
        return n
    table = [0] * (n + 1)
    table[1] = 1
    for index in range(2, n + 1):
        table[index] = table[index - 1] + table[index - 2]
    return table[n]

print("F(100) =", fibo_bottom_up(100))

## Optimiser l’espace

La prochaine valeur utilise seulement les deux précédentes. Le temps reste O(n), mais l’espace auxiliaire devient O(1).

In [ ]:
def fibo_constant_space(n):
    if n <= 1:
        return n
    previous, current = 0, 1
    for _ in range(2, n + 1):
        previous, current = current, previous + current
    return current

print("F(100) =", fibo_constant_space(100))

## Comparer les versions

| Version | Temps | Espace auxiliaire |
|---|---:|---:|
| récursive naïve | O(2 puissance n), borne simple | O(n) |
| mémoïsée | O(n) | O(n) |
| Bottom-Up avec table | O(n) | O(n) |
| deux variables | O(n) | O(1) |

Cette analyse compte chaque addition comme une opération. Pour des entiers gigantesques, le nombre de chiffres augmente et l’addition elle-même devient plus coûteuse : c’est la complexité en bits, hors du périmètre débutant.

In [ ]:
n = 28
print("naïve   :", timeit.timeit(lambda: fibo_naive(n), number=1))
print("mémoïsée:", timeit.timeit(lambda: fibo_memo(n), number=1))
print("table   :", timeit.timeit(lambda: fibo_bottom_up(n), number=1))
print("2 vars  :", timeit.timeit(lambda: fibo_constant_space(n), number=1))

## Quand penser à la programmation dynamique ?

- Les mêmes sous-problèmes reviennent plusieurs fois.
- La solution globale peut être construite à partir de solutions plus petites.

Fibonacci illustre surtout le premier point. Les problèmes de chemins et de choix optimaux montrent davantage le second.

## À vous de jouer — l’escalier

On monte une ou deux marches à la fois. ways(n)=ways(n-1)+ways(n-2), avec ways(0)=1 et ways(1)=1. Écrivez une version itérative avec deux variables et annoncez ses complexités.

In [ ]:
def nombre_facons_escalier(n):
    if n < 0:
        raise ValueError("n doit être positif")
    previous, current = 1, 1
    for _ in range(2, n + 1):
        previous, current = current, previous + current
    return current

for n in range(8):
    print(n, "marches →", nombre_facons_escalier(n), "façon(s)")

# 6. Mesurer sans confondre mesure et complexité

Big O fournit un modèle théorique. timeit, perf_counter et cProfile observent une exécution réelle. Les deux approches sont complémentaires.

![Big O et profiling](../../ressources/illustrations/jour_01/10_theorie_mesure.png)

Un benchmark sérieux répète l’opération, contrôle les données et compare des fonctions produisant le même résultat. Une mesure unique peut être perturbée par le système ou le cache.

## Mesurer avec timeit

timeit répète une petite opération. Il ne devine pas la complexité : il faut tester plusieurs tailles et interpréter la forme de la croissance.

In [ ]:
def sum_python_loop(values):
    total = 0
    for value in values:
        total += value
    return total

for n in [10_000, 20_000, 40_000, 80_000]:
    values = list(range(n))
    elapsed = timeit.timeit(lambda: sum_python_loop(values), number=20)
    print(f"n={n:>6,} → {elapsed:.6f} s")

## Trouver les goulots avec cProfile

cProfile mesure chaque fonction. ncalls est le nombre d’appels, tottime le temps hors sous-appels et cumtime le temps incluant les sous-appels. Le tri cumulative révèle les responsables du temps total.

In [ ]:
def linear_work(n):
    return sum(range(n))

def quadratic_work(n):
    total = 0
    for left in range(n):
        for right in range(n):
            total += left == right
    return total

profiler = cProfile.Profile()
profiler.enable()
linear_work(100_000)
quadratic_work(500)
profiler.disable()
stream = io.StringIO()
pstats.Stats(profiler, stream=stream).sort_stats("cumulative").print_stats(10)
print(stream.getvalue())

## Limites de la mesure

Le profiler ajoute un petit coût et les données de démonstration peuvent différer de la production. Choisissez d’abord un algorithme adapté, mesurez ensuite, puis optimisez le véritable goulot sans détruire inutilement la lisibilité.

# 7. Mini-projet — détecter un doublon

Comparer chaque paire prend O(n²) et O(1) d’espace. Mémoriser les valeurs vues dans un set prend O(n) en moyenne et O(n) d’espace : on échange de la mémoire contre du temps.

In [ ]:
def has_duplicate_quadratic(values):
    for left in range(len(values)):
        for right in range(left + 1, len(values)):
            if values[left] == values[right]:
                return True
    return False

def has_duplicate_with_set(values):
    seen = set()
    for value in values:
        if value in seen:
            return True
        seen.add(value)
    return False

print(has_duplicate_quadratic([4, 7, 2, 9, 7]))
print(has_duplicate_with_set([4, 7, 2, 9, 7]))

## À vous de jouer — retourner le premier doublon

Complétez first_duplicate pour retourner le premier doublon rencontré, ou None. Utilisez un set et annoncez ses complexités moyennes en temps et en espace.

In [ ]:
def first_duplicate(values):
    seen = set()
    for value in values:
        if value in seen:
            return value
        seen.add(value)
    return None

print(first_duplicate([4, 7, 2, 9, 7, 3]))
print(first_duplicate([1, 2, 3, 4]))

In [ ]:
for n in [250, 500, 1_000, 2_000]:
    values = list(range(n))
    quadratic = timeit.timeit(lambda: has_duplicate_quadratic(values), number=1)
    hashed = timeit.timeit(lambda: has_duplicate_with_set(values), number=1)
    print(f"n={n:>5,} | paires={quadratic:.6f} s | set={hashed:.6f} s")

## À vous de jouer — profiler les deux versions

Profilez les deux fonctions sur list(range(1_500)), triez les résultats par temps cumulatif et affichez les huit premières lignes.

In [ ]:
exercise_profiler = cProfile.Profile()
exercise_profiler.enable()
has_duplicate_quadratic(list(range(1_500)))
has_duplicate_with_set(list(range(1_500)))
exercise_profiler.disable()
exercise_stream = io.StringIO()
pstats.Stats(exercise_profiler, stream=exercise_stream).sort_stats("cumulative").print_stats(8)
print(exercise_stream.getvalue())

# 8. Fiche mentale

1. définir ce que représente n ;
2. préciser meilleur, moyen ou pire cas ;
3. repérer les opérations dépendant de n ;
4. additionner les blocs successifs ;
5. multiplier les boucles imbriquées ;
6. reconnaître les divisions répétées ;
7. garder le terme dominant ;
8. analyser temps et mémoire séparément ;
9. vérifier les préconditions ;
10. mesurer les vrais goulots après l’analyse.

## Vérification finale

Vous devez pouvoir expliquer pourquoi log2(1 000 000) vaut environ 20, pourquoi la recherche change entre list et set, pourquoi trier pour une seule recherche peut être inutile, pourquoi Fibonacci naïf refait du travail et pourquoi Big O et cProfile ne donnent pas la même information.

## À retenir

La complexité n’est pas une liste de symboles à mémoriser. Elle permet de prévoir ce qui arrivera lorsque les données grandiront. Reliez la structure du code au travail répété, puis choisissez la structure ou la stratégie qui évite ce travail inutile.